# Introduction to self-supervised learning with basic methods

## 20/03/2026    

### a.y. 2025/2026


In [ ]:
import numpy as np
from tensorflow import keras
import random
import matplotlib.pyplot as plt
from keras.applications.densenet import DenseNet121
from keras.applications.densenet import preprocess_input
from keras.layers import GlobalAveragePooling2D,Dense
from keras.models import Model
import tensorflow as tf
!mkdir checkpoints

In this lab, we will understand better the working principles and the benefits of self-supervised learning.
As an example, we will utilize the dataset CIFAR10, which is available in *keras datasets*.


In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

#dictionary for the classes

classes = {0 :"airplane",
1	:"automobile",
2	:"bird",
3	:"cat",
4	:"deer",
5	:"dog",
6	:"frog",
7	:"horse",
8	:"ship",
9	:"truck"}

Let us visualize ten random samples from CIFAR10

In [ ]:
#pick ten random training images (using the library random) and visualize them in a grid with matplotlib
samples = # fill here

Now, we will initiliaze a model based on DenseNet121 (from scratch), a good trade-off between efficiency and efficacy.
We will use the library keras applications.

In [ ]:
def initialize_model(num_classes, learning_rate=0.001):
    base_model = DenseNet121(weights=None, include_top=False)

    x = base_model.output

    #now, we will build the model for CIFAR10 classification

    # let's add a Global average pooling
    x = # fill here

    # let's add a fully-connected layer
    x = # fill here

    # let's add the output layer
    preds = # fill here

    # define the full model by linking inputs and outputs
    model = Model(inputs=base_model.input, outputs=preds)

    #initialize the optimizer
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

    # Compile the model, passing the optimizer as parameter.
    # Also set the loss to be "categorical_crossentropy"
    model.compile(
        optimizer=# fill here,
        loss=# fill here,
        metrics=["accuracy"]
    )

    return model


Now, normalize input images to be floating point values between 0 and 1, then convert labels to one-hot encoding (using `keras.utils.to_categorical`). Be sure to apply normalization on both train and test data.

In [ ]:
#normalize x_train
x_train = x_train/255.0
#categorize y_train
y_train = keras.utils.to_categorical(y_train)

# Do the same for the test set
x_test = # fill here
y_test = # fill here

Now, create the three subsets with cardinality c1 = 2000, c2= 5000, c3 = 10000.

**tip**: we want a balanced dataset, so use stratified train test split from sklearn library (the idea will be to create a split with ${test\_size}_i = (1 - \frac{c\_i}{\vert x\_train \vert}, i=1,2,3$) discarding the test split.

In [ ]:
from sklearn.model_selection import train_test_split

# for one subset
# fill here = train_test_split(x_train, y_train, test_size=2000, stratify= # fill here)


**Initialize one model for each subset and store them, so that you can evaluate them separately**

use `model.save_weights("checkpoints/model_name.weights.h5")` to store the learned weights.

Choose a different checkpoint name for each model.


In [ ]:
for i, (x_train_subset, y_train_subset) in enumerate(subsets):
    model = # fill here
    model.fit(# fill here)
    model.save_weights(# fill here.weights.h5")


You can load a specific model, you can use `model.load_weights("checkpoints/model_name.weights.h5")`

Evaluate each model on the whole **original** test set, computing its accuracy.

Recall that accuracy

In [ ]:
model.load_weights(f"checkpoints/model_subset_0.weights.h5")
acc_2000 =  # fill here

Now, we will implement a rotation self-supervised pre-text task.
So:



*   Create the pre-text dataset
*   Initialize a model
*   train the model from scratch on the pre-text task



**Create the pre-text dataset**

input: a training set with no labels (x_train,), rotation_set = [0,1,2,3]

create a copy or original dataset (x_train_rotated)

create an empty vector of labels (y_train_rotated)

```
For i in range(0,len(x_train)):

    k = pick a random element of rotation set

    #look for an appropriate function in tf.image

    y_train_rotated[i] = k

```

In [ ]:
# Generate random rotation labels for the entire dataset at once
y_train_rotated = # fill here

# Apply rotations to all images having the same "rotation label"
def rotate_batch(images, rotations):
    rotated_images = np.empty_like(images)  # Preallocate memory

    for k in range(4):  # Loop over rotation angles (0, 1, 2, 3)
        mask = # Boolean mask for this rotation
        rotated_images[mask] = # rotates all images having the same y_train_rotated label, look for the appropriate function in tf.image

    return rotated_images

x_train_rotated = rotate_batch(x_train, y_train_rotated)




You can store and/or load your new dataset to save time and computation. You have an empty cell below to do so if you want

```python
np.save("checkpoints/x_train_rotated.npy", x_train_rotated)
np.save("checkpoints/y_train_rotated.npy", y_train_rotated)
x_train_rotated = np.load("checkpoints/x_train_rotated.npy")
y_train_rotated = np.load("checkpoints/y_train_rotated.npy")
```

Visualize ten random samples to verify the obtained dataset

In [ ]:
# code here

Re-define a DenseNet121 model, as done before (**careful on the required number of classes**).

In [ ]:
# model = fill here

fit the model for the pre-text task

In [ ]:
#normalize the images in x_train_rotated
x_train_rotated = # fill here
# then convert pretext task labels (y_train_rotated) to one-hot encoding
y_train_rotated = # fill here

#fit the model and store its weights, like you did before

# fill here


In [ ]:
# It shows a summary of the model
model.summary()

Now, we are ready to fine-tune the model on the downstream task.

To do so:

-   Load the trained backbone of the pre-text task
-   Re-define the classification hat (with the appropriate output layer):
    + Take inspiration from the function `initialize_model`, and consider that than you can take the features of the pretext model coming out from `model.layers[-4].output`.
    + Then, add `GlobalAveragePooling2D()`, a Dense layer with 512 neurons + `relu` activation, and a prediction layer with the correct number of outputs and softmax activation
-   Re-train (use a lower learning rate than training from scratch)



In [ ]:
model.load_weights(# fill here
x = # fill here
x = # fill here
x = # fill here
preds = # fill here
model = Model(inputs=model.input, outputs=preds)

# Initialize the optimizer with a learning rate of 0.00001
optimizer = # fill here

# Compile the model as done in "initialize_model" function
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=["accuracy"])

# Fit the model on the subset
model.fit(# fill here

# Save the model
model.save_weights(# fill here

In [ ]:
#evaluate the model on the whole original test set

Make a plot of (scratch_accuracy and SSL_accuracy) with the three different subset sizes.

Do you get any benefit on the downstream task, with self-supervision?

Comment on the results

**Optional** If you have time, you can repeat the previous experiments also with the full dataset.
